To extract the unique categories from the OSM and Foursquare datasets, you can use the following SQL commands. These commands create new tables containing distinct category information and export them to CSV files.

```sql
CREATE TABLE osm_categories AS
SELECT DISTINCT osm_class, osm_type FROM osm ;
\copy osm_categories TO 'osm_categories.csv' WITH ( FORMAT csv, HEADER, DELIMITER ',', QUOTE '"', ESCAPE '"', NULL '' );
```


Foursquare can be downloaded from the Foursquare website: https://docs.foursquare.com/data-products/docs/categories
(data/fsq_personalization-apis-movement-sdk-categories.csv)

Then we feed data/fsq_personalization-apis-movement-sdk-categories.csv, osm_categories.csv and https://wiki.openstreetmap.org/wiki/Map_features to `GPT 5.5 with High Intelligence` to generate a mapping between Foursquare categories and OSM categories. Resulted in a mapping file fsq_osm_category_mapping.csv. 

prompt: "I want you to add two columns to fsq_personalization-apis-movement-sdk-categories.csv that are osm_class, osm_type that they have the same meaning. If ou cannot find a match from osm please put nf_fsq for both osm_class, osm_type."

In [ ]:
import pandas
fsq_osm_category_mapping = pandas.read_csv('../../data/fsq_osm_category_mapping.csv')
fsq_personalization = pandas.read_csv('../../data/fsq_personalization-apis-movement-sdk-categories.csv')

In [ ]:

# check that the two files have the same [Category ID, Category Name, Category Label] values 
f1 = fsq_osm_category_mapping[['Category ID', 'Category Name', 'Category Label']].sort_values(by=['Category ID', 'Category Name', 'Category Label']).reset_index(drop=True)
f2 = fsq_personalization[['Category ID', 'Category Name', 'Category Label']].sort_values(by=['Category ID', 'Category Name', 'Category Label']).reset_index(drop=True)
assert f1.equals(f2), "The two files do not have the same [Category ID, Category Name, Category Label] values. Please check the files for discrepancies."

In [ ]:
category_mapping_dict = {}
for index, row in fsq_osm_category_mapping.iterrows():
    category_mapping_dict[row['Category ID']] = {
                                            'Category Name': row['Category Name'],
                                            'Category Label': row['Category Label'],
                                            "osm_class": row['osm_class'],
                                             "osm_type": row['osm_type']}


In [ ]:
world_poi_df = pandas.read_csv('../../data/World_POI_levenshtein_0.3.csv', chunksize=100_000)

In [ ]:
def is_similar(category_dict, row):
    res=False
    if row['osm_class'] in category_dict['osm_class']:
        res = True
    elif row['osm_type'] in category_dict['osm_type']:
        res = True
                
    return res
                        

for chunk in world_poi_df:
    # print(chunk.columns)
    chunk.dropna(subset=['fsq_category_ids'], inplace=True)
    for index, row in chunk.iterrows():
        if row['fsq_category_ids'] != '':
            row['fsq_category_ids'] = row['fsq_category_ids'].replace(' ', ', ')
            cat_ids = eval(row['fsq_category_ids'])
            row['semantic_similarity'] = 0
            for cat_id in cat_ids:
                if cat_id in category_mapping_dict.keys():
                    if is_similar(category_mapping_dict[cat_id], row):
                        row['semantic_similarity'] = 1
